# 07 — Final Comparison and Validation

This notebook compares the saved Adaptive v3 and CP-SAT schedules, checks graph conflicts, validates selected benchmark constraints, and measures same-day adjacent-period student-gap violations.

**Important:** this is a comparison/validation notebook. It does not rebuild or solve the CP-SAT model. The current saved schedules are expected to show gap violations until the solver model is updated.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import networkx as nx
from IPython.display import display

# Works when launched from either the project root or src/
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "data" / "processed").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/processed. Open this notebook from the project root or src/.")

PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw" / "Nott"

required_paths = [
    PROCESSED / "conflict_graph.gpickle",
    PROCESSED / "adaptive_v3_schedule.csv",
    PROCESSED / "cp_sat_schedule.csv",
    RAW / "enrolements",
]
for path in required_paths:
    print(("FOUND " if path.exists() else "MISSING ") + str(path))
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Required project files are missing:\n" + "\n".join(missing))
print("\nProject root:", PROJECT_ROOT)

## 2. Load conflict graph, schedules, and enrolments

In [ ]:
with open(PROCESSED / "conflict_graph.gpickle", "rb") as f:
    G = pickle.load(f)

def load_schedule(path):
    df = pd.read_csv(path)
    required = {"exam_id", "period_index", "date"}
    absent = required - set(df.columns)
    if absent:
        raise ValueError(f"{path.name} is missing columns: {sorted(absent)}")
    df = df.copy()
    df["exam_id"] = df["exam_id"].astype(str).str.strip()
    df["period_index"] = pd.to_numeric(df["period_index"], errors="raise").astype(int)
    df["date"] = pd.to_datetime(df["date"], errors="raise").dt.date
    if df["exam_id"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate exam IDs.")
    return df

adaptive_v3 = load_schedule(PROCESSED / "adaptive_v3_schedule.csv")
cp_sat = load_schedule(PROCESSED / "cp_sat_schedule.csv")

enrol_raw = pd.read_csv(RAW / "enrolements", sep=r"\\s+", header=None, dtype=str, engine="python")
if enrol_raw.shape[1] < 2:
    raise ValueError("The enrolments file must contain at least student ID and exam ID columns.")
enrolments = enrol_raw.iloc[:, [0, 1]].copy()
enrolments.columns = ["student_id", "exam_id"]
enrolments = enrolments.dropna()
enrolments["student_id"] = enrolments["student_id"].astype(str).str.strip()
enrolments["exam_id"] = enrolments["exam_id"].astype(str).str.strip()
enrolments = enrolments.drop_duplicates()

print(f"Conflict graph: {G.number_of_nodes():,} exams, {G.number_of_edges():,} edges")
print(f"Raw enrolment rows: {len(enrol_raw):,}; unique student-exam pairs: {len(enrolments):,}")
display(pd.DataFrame([
    {"Algorithm": "Adaptive v3", "Schedule rows": len(adaptive_v3), "Unique exams": adaptive_v3.exam_id.nunique(), "Periods used": adaptive_v3.period_index.nunique()},
    {"Algorithm": "CP-SAT", "Schedule rows": len(cp_sat), "Unique exams": cp_sat.exam_id.nunique(), "Periods used": cp_sat.period_index.nunique()},
]))

## 3. Check schedule completeness and direct graph conflicts

In [ ]:
graph_exams = {str(x).strip() for x in G.nodes()}

def validate_graph_schedule(schedule, algorithm):
    period_map = dict(zip(schedule["exam_id"], schedule["period_index"]))
    scheduled = set(period_map)
    missing = graph_exams - scheduled
    extra = scheduled - graph_exams
    conflicts = [
        (str(u), str(v)) for u, v in G.edges()
        if str(u).strip() in period_map
        and str(v).strip() in period_map
        and period_map[str(u).strip()] == period_map[str(v).strip()]
    ]
    return {
        "Algorithm": algorithm,
        "Graph exams": len(graph_exams),
        "Scheduled unique exams": len(scheduled),
        "Missing graph exams": len(missing),
        "Extra exams": len(extra),
        "Periods used": schedule["period_index"].nunique(),
        "Direct graph-conflict violations": len(conflicts),
        "Graph-feasible": len(missing) == 0 and len(extra) == 0 and len(conflicts) == 0,
    }, conflicts

adaptive_graph_check, adaptive_graph_conflicts = validate_graph_schedule(adaptive_v3, "Adaptive v3")
cp_graph_check, cp_graph_conflicts = validate_graph_schedule(cp_sat, "CP-SAT")
graph_validation = pd.DataFrame([adaptive_graph_check, cp_graph_check])
display(graph_validation)
if adaptive_graph_conflicts:
    print("Adaptive v3 conflict examples:", adaptive_graph_conflicts[:10])
if cp_graph_conflicts:
    print("CP-SAT conflict examples:", cp_graph_conflicts[:10])

## 4. Measure student one-period-gap violations

The benchmark says students must have a one-period (or overnight) gap between consecutive exams. For this check, a violation is a pair of distinct exams taken by the same student that are scheduled on the **same date** in consecutive period indices.

This validator deduplicates repeated student–exam rows and does not count overnight pairs as violations. It measures the saved schedules; it does not fix them.

In [ ]:
def find_student_gap_violations(schedule, enrolments):
    s = schedule[["exam_id", "period_index", "date"]].copy()
    s["exam_id"] = s["exam_id"].astype(str).str.strip()
    s["period_index"] = pd.to_numeric(s["period_index"], errors="raise").astype(int)
    s["date"] = pd.to_datetime(s["date"], errors="raise").dt.date

    e = enrolments[["student_id", "exam_id"]].drop_duplicates().copy()
    merged = e.merge(s, on="exam_id", how="left", validate="many_to_one", indicator=True)
    missing = merged.loc[merged["_merge"] != "both", "exam_id"].drop_duplicates().tolist()
    if missing:
        raise ValueError(f"{len(missing)} enrolled exam IDs are not scheduled. Examples: {missing[:10]}")
    merged = merged.drop(columns="_merge")

    violations = []
    affected_students = set()
    for student_id, group in merged.groupby("student_id", sort=False):
        group = group.drop_duplicates("exam_id")
        records = list(group[["exam_id", "period_index", "date"]].itertuples(index=False, name=None))
        for i in range(len(records)):
            exam_a, period_a, date_a = records[i]
            for j in range(i + 1, len(records)):
                exam_b, period_b, date_b = records[j]
                if date_a == date_b and abs(period_a - period_b) == 1:
                    affected_students.add(student_id)
                    violations.append({
                        "student_id": student_id,
                        "exam_1": exam_a,
                        "period_1": period_a,
                        "exam_2": exam_b,
                        "period_2": period_b,
                        "date": date_a,
                    })
    violation_df = pd.DataFrame(violations, columns=[
        "student_id", "exam_1", "period_1", "exam_2", "period_2", "date"
    ])
    summary = {
        "Affected students": len(affected_students),
        "Violating student-exam pairs": len(violation_df),
        "Gap rule passes": len(violation_df) == 0,
    }
    return summary, violation_df

adaptive_gap_summary, adaptive_gap_detail = find_student_gap_violations(adaptive_v3, enrolments)
cp_gap_summary, cp_gap_detail = find_student_gap_violations(cp_sat, enrolments)

gap_validation = pd.DataFrame([
    {"Algorithm": "Adaptive v3", **adaptive_gap_summary},
    {"Algorithm": "CP-SAT", **cp_gap_summary},
])
display(gap_validation)
print("Adaptive v3 examples:")
display(adaptive_gap_detail.head(10))
print("CP-SAT examples:")
display(cp_gap_detail.head(10))

## 5. Combine the validation results

In [ ]:
validation_summary = graph_validation.merge(gap_validation, on="Algorithm", how="left")
validation_summary["All checks in this notebook pass"] = (
    validation_summary["Graph-feasible"] & validation_summary["Gap rule passes"]
)
display(validation_summary)

## 6. Create a concise comparison table

The adjacent-exam gap count is reported as a separate measure, not as proof that the schedule is fully benchmark-compliant. The percentage reduction compares CP-SAT with Adaptive v3 only when the Adaptive v3 penalty is non-zero.

In [ ]:
comparison = pd.DataFrame([
    {
        "Algorithm": "Adaptive v3",
        "Periods used": int(adaptive_v3["period_index"].nunique()),
        "Direct graph-conflict violations": int(adaptive_graph_check["Direct graph-conflict violations"]),
        "Affected students (gap rule)": int(adaptive_gap_summary["Affected students"]),
        "Adjacent-period violation pairs": int(adaptive_gap_summary["Violating student-exam pairs"]),
        "Gap rule passes": bool(adaptive_gap_summary["Gap rule passes"]),
    },
    {
        "Algorithm": "CP-SAT",
        "Periods used": int(cp_sat["period_index"].nunique()),
        "Direct graph-conflict violations": int(cp_graph_check["Direct graph-conflict violations"]),
        "Affected students (gap rule)": int(cp_gap_summary["Affected students"]),
        "Adjacent-period violation pairs": int(cp_gap_summary["Violating student-exam pairs"]),
        "Gap rule passes": bool(cp_gap_summary["Gap rule passes"]),
    },
])
adaptive_penalty = int(adaptive_gap_summary["Violating student-exam pairs"])
cp_penalty = int(cp_gap_summary["Violating student-exam pairs"])
if adaptive_penalty:
    cp_reduction_vs_adaptive_pct = 100 * (adaptive_penalty - cp_penalty) / adaptive_penalty
else:
    cp_reduction_vs_adaptive_pct = np.nan
print(f"CP-SAT gap-violation change relative to Adaptive v3: {cp_reduction_vs_adaptive_pct:.2f}% (positive means fewer violations)")
display(comparison)

## 7. Export validation outputs

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
comparison_path = PROCESSED / "final_algorithm_comparison.csv"
validation_path = PROCESSED / "final_schedule_validation.csv"
adaptive_detail_path = PROCESSED / "adaptive_v3_student_gap_violations.csv"
cp_detail_path = PROCESSED / "cp_sat_student_gap_violations.csv"

comparison.to_csv(comparison_path, index=False)
validation_summary.to_csv(validation_path, index=False)
adaptive_gap_detail.to_csv(adaptive_detail_path, index=False)
cp_gap_detail.to_csv(cp_detail_path, index=False)

print("Saved:")
for path in [comparison_path, validation_path, adaptive_detail_path, cp_detail_path]:
    print(" -", path)

## 8. What the results mean and what to do next

- **Direct graph conflicts = 0** means no graph-connected pair of exams shares a period.
- **Gap violations > 0** means the stronger student one-period-gap requirement is not satisfied, even if graph conflicts are zero.
- The current saved results previously showed 995 gap-violation pairs for Adaptive v3 and 4,005 for CP-SAT. This notebook recomputes those values from the schedule files and the deduplicated source enrolments.
- **Do not report either schedule as fully compliant yet.**
- The next coding change belongs in the notebook/script that actually creates `cp_sat_schedule.csv`, not in this comparison notebook. Add hard constraints for each student's exam pairs so that exams cannot occupy adjacent valid periods on the same day; then solve, export the new schedule, and rerun this notebook.
- Use the actual calendar date/time mapping to define adjacency. Do not simply treat any consecutive global period indices as adjacent across dates.
- Room allocation/capacity, fixed rooms, earliness priorities, and other special benchmark requirements also need separate implementation and validation before claiming full Nottingham benchmark compliance.